In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
RANDOM_STATE = 42

In [ ]:
# Load the cluster data
df_cluster2 = pd.read_csv("cluster_2.csv")
print("Data loaded. Shape:", df_cluster2.shape)

Data loaded. Shape: (2059, 98)


In [ ]:
# Loading the cluster outputs
top40 = joblib.load("top_features_for_clustering.joblib")
features_to_use = top40
X_sub = df_cluster2[features_to_use].copy()
y_sub = df_cluster2["Bankrupt?"].copy()
print("Cluster 2 shape:", X_sub.shape)
print("Target Distribution:\n", y_sub.value_counts())

Cluster 2 shape: (2059, 40)
Target Distribution:
 Bankrupt?
0    2053
1       6
Name: count, dtype: int64


In [ ]:
# Base models that handles imbalanced data
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
et = ExtraTreesClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
hgb = HistGradientBoostingClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE
)
base_estimators = [
    ('rf', rf),
    ('et', et),
    ('hgb', hgb)
]

In [ ]:
# Defining meta model
meta_model = LogisticRegression(
    penalty="l2",
    class_weight="balanced",
    random_state=RANDOM_STATE
)
# Using StratifiedKFold with cv=3 since Cluster 2 has only six positives, if cv=5, some folds may get zero or one positive sample, making training unstable.
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1
)
#Scale the data first, then Stack
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('stacking', stacking_clf)
])

In [ ]:
# Fitting the model
print("Fitting model...")
model_pipeline.fit(X_sub, y_sub)
# Predict on the original cluster's train rows
y_pred = model_pipeline.predict(X_sub)
#Confusion matrix & Eq(1) accuracy
cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
# True Bankrupts Caught (True Positives)
TT = int(tp)
# Bankrupts Missed (False Negatives)
TF = int(fn)
N_features = len(features_to_use)
# Eq (1) accuracy: TT / (TF + TT)
eq1_acc = TT / (TF + TT) if (TF + TT) > 0 else 0
print("-" * 40)
print("RESULTS FOR TABLE 3 (Cluster 2 Model)")
print("-" * 40)
print(f"Confusion Matrix:\n{cm}")
print(f"TT (True Bankrupts Caught): {TT}")
print(f"TF (Bankrupts Missed):      {TF}")
print(f"N_features:                 {N_features}")
print(f"Eq(1) Accuracy:             {eq1_acc:.4f}")
print("-" * 40)

Fitting model...
----------------------------------------
RESULTS FOR TABLE 3 (Cluster 2 Model C)
----------------------------------------
Confusion Matrix:
[[2013   40]
 [   0    6]]
TT (True Bankrupts Caught): 6
TF (Bankrupts Missed):      0
N_features:                 40
Eq(1) Accuracy:             1.0000
----------------------------------------


In [ ]:
cluster2_package = {
    "cluster_id": 2,
    "features": features_to_use,
    "pipeline": model_pipeline, # For both scaler and stacking model
    "table3_stats": {"TT": TT, "TF": TF, "Eq1_acc": eq1_acc, "N_features": N_features}
}

joblib.dump(cluster2_package, "cluster2_stacking_C.joblib")
print("Saved cluster2_stacking_C.joblib")

Saved cluster2_stacking_C.joblib
